# Summary HTML

Generate an HTML summary report for the current Prognosis final-selection results.

In [1]:
# ============================================================
# 1. Imports and settings
# ============================================================

import html
import json
import os
import subprocess
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

model_root = Path('/host/d/projects/Habitats/models/Prognosis')
project_root = Path('/host/d/projects/Habitats')
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

asset_dir = project_root / 'summary_assets'
asset_dir.mkdir(parents=True, exist_ok=True)

summary_results_path = results_dir / 'summary.html'
summary_root_path = project_root / 'summary.html'

branches = [
    {'name': 'Clinical', 'folder': model_root / 'clinical' / 'final_selections', 'description': 'Final-selected clinical-variable model.'},
    {'name': 'Whole-image radiomics', 'folder': model_root / 'whole_image' / 'final_selections', 'description': 'Final-selected conventional whole-tumor radiomics model.'},
    {'name': 'Habitat radiomics', 'folder': model_root / 'habitats_sum' / 'final_selections', 'description': 'Final-selected fixed-K habitat-sum radiomics model.'},
    {'name': 'DL 3D ML', 'folder': model_root / 'dl_3d_ml_all' / 'final_selections', 'description': 'Final-selected ML model based on 3D deep-learning features.'},
]

fusion_soft_vote = {
    'name': 'Fusion soft vote',
    'metrics_path': model_root / 'fusion' / 'soft_vote_metrics.xlsx',
    'manifest_path': model_root / 'fusion' / 'soft_vote_manifest.json',
    'roc_paths': {
        'train': model_root / 'fusion' / 'ROC_curve_soft_vote_cv.pdf',
        'internal test': model_root / 'fusion' / 'ROC_curve_soft_vote_internal_test.pdf',
        'external test': model_root / 'fusion' / 'ROC_curve_soft_vote_external_test.pdf',
    },
}

stacking_final_root = model_root / 'fusion' / 'stacking' / 'final_selections'
# Paper version: RF is the selected fusion stacking algorithm; KNN is excluded.
selected_stacking_methods = ['RF']
stacking_folders = [
    stacking_final_root / method
    for method in selected_stacking_methods
    if (stacking_final_root / method).is_dir()
    and ((stacking_final_root / method) / 'cv_final_selection_metrics.xlsx').is_file()
] if stacking_final_root.is_dir() else []

dataset_specs = [
    ('train', 'cv_final_selection_metrics.xlsx', 'ROC_curve_cv_final_selection.pdf'),
    ('internal test', 'internal_test_final_selection_metrics.xlsx', 'ROC_curve_internal_test_final_selection.pdf'),
    ('external test', 'external_test_final_selection_metrics.xlsx', 'ROC_curve_external_test_final_selection.pdf'),
]

print('Model root:', model_root)
print('Summary output:', summary_results_path)
print('Convenience copy:', summary_root_path)
print('Stacking final-selection folders:', [p.name for p in stacking_folders])


Model root: /host/d/projects/Habitats/models/Prognosis
Summary output: /host/d/projects/Habitats/results/summary.html
Convenience copy: /host/d/projects/Habitats/summary.html
Stacking final-selection folders: ['RF']


In [2]:
# ============================================================
# 2. Helper functions
# ============================================================

def esc(x):
    return html.escape(str(x))


def read_json(path):
    path = Path(path)
    if not path.is_file():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def fmt3(x):
    if x is None or pd.isna(x):
        return 'NA'
    return f'{float(x):.3f}'


def fmt_ci(row):
    if 'auc_ci_low' not in row.index or 'auc_ci_high' not in row.index:
        return 'NA'
    if pd.isna(row['auc_ci_low']) or pd.isna(row['auc_ci_high']):
        return 'NA'
    return f"{float(row['auc_ci_low']):.3f}-{float(row['auc_ci_high']):.3f}"


def normalize_algorithm_name(name):
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ''
    text = str(name).strip()
    lookup = {
        'xgboost': 'XGBoost',
        'xgboosh': 'XGBoost',
        'svm': 'SVM',
        'lr': 'LR',
        'rf': 'RF',
        'knn': 'KNN',
        'soft_vote': 'N/A',
        'soft_vote_eq': 'N/A',
    }
    return lookup.get(text.lower(), text)


def classifier_from_setting_tag(setting_tag):
    if setting_tag is None or (isinstance(setting_tag, float) and np.isnan(setting_tag)):
        return None
    text = str(setting_tag)
    if '__' in text:
        return text.split('__', 1)[0]
    return text.split('_', 1)[0]


def selected_settings_from_manifest(manifest):
    if manifest is None:
        return []
    for key in ['selected_cv_experiments', 'selected_experiments', 'selected_settings', 'cv_selected_settings', 'settings']:
        value = manifest.get(key)
        if isinstance(value, list) and len(value) > 0:
            return value
    return []


def majority_algorithm_from_manifest(manifest, fallback='NA'):
    settings = selected_settings_from_manifest(manifest)
    algos = []
    for item in settings:
        if isinstance(item, dict):
            algo = item.get('classifier') or item.get('method') or item.get('algorithm')
            if algo is None:
                algo = classifier_from_setting_tag(item.get('setting_tag') or item.get('experiment'))
        else:
            algo = classifier_from_setting_tag(item)
        if algo:
            algos.append(normalize_algorithm_name(algo))
    if len(algos) == 0:
        return normalize_algorithm_name(fallback)
    counts = Counter(algos)
    best_algo = None
    best_count = -1
    for algo in algos:
        if counts[algo] > best_count:
            best_algo = algo
            best_count = counts[algo]
    return best_algo


def dataframe_to_html(df, index=False):
    return df.to_html(index=index, escape=True, border=0, classes='dataframe')


def describe_settings(settings):
    if len(settings) == 0:
        return '<p class="muted">No selected-setting details found in manifest.</p>'
    rows = []
    for item in settings:
        if isinstance(item, dict):
            classifier = item.get('classifier', item.get('method', item.get('algorithm', '')))
            experiment = item.get('experiment', item.get('setting_tag', item.get('name', '')))
            extra = {k: v for k, v in item.items() if k not in ['classifier', 'method', 'algorithm', 'experiment', 'setting_tag', 'name']}
            rows.append({
                'Classifier': normalize_algorithm_name(classifier),
                'Experiment': experiment,
                'Details': ', '.join([f'{k}={v}' for k, v in extra.items()]) if extra else '',
            })
        else:
            rows.append({'Classifier': normalize_algorithm_name(classifier_from_setting_tag(item)), 'Experiment': item, 'Details': ''})
    return dataframe_to_html(pd.DataFrame(rows), index=False)


def read_final_metric_row(folder, metric_filename):
    path = Path(folder) / metric_filename
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_excel(path)
    if 'is_final_selection' in df.columns:
        final_df = df[df['is_final_selection'].astype(bool)].copy()
        if final_df.shape[0] > 0:
            return final_df.iloc[-1], path, df
    return df.iloc[-1], path, df


def collect_final_selection_metrics(folder):
    rows = []
    source_rows = []
    for dataset_name, metric_file, _ in dataset_specs:
        row, path, _ = read_final_metric_row(folder, metric_file)
        rows.append({
            'Dataset': dataset_name,
            'AUC': fmt3(row.get('auc')),
            '95% CI': fmt_ci(row),
            'Accuracy': fmt3(row.get('accuracy')),
            'Sensitivity': fmt3(row.get('sensitivity')),
            'Specificity': fmt3(row.get('specificity')),
            'Probability source': row.get('probability_column', ''),
        })
        source_rows.append({'Dataset': dataset_name, 'Metric file': str(path), 'Probability source': row.get('probability_column', '')})
    return pd.DataFrame(rows), pd.DataFrame(source_rows)


def collect_soft_vote_metrics(metrics_path):
    df = pd.read_excel(metrics_path)
    dataset_map = {'train': 'cv', 'internal test': 'internal_test', 'external test': 'external_test'}
    rows = []
    for display_name, source_name in dataset_map.items():
        row_df = df[df['dataset'].astype(str) == source_name]
        if row_df.shape[0] == 0:
            raise RuntimeError(f'Missing soft-vote dataset: {source_name}')
        row = row_df.iloc[-1]
        rows.append({
            'Dataset': display_name,
            'AUC': fmt3(row.get('auc')),
            '95% CI': fmt_ci(row),
            'Accuracy': fmt3(row.get('accuracy')),
            'Sensitivity': fmt3(row.get('sensitivity')),
            'Specificity': fmt3(row.get('specificity')),
            'Probability source': row.get('probability_column', ''),
        })
    return pd.DataFrame(rows)


def pdf_to_png(pdf_path, output_stem):
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        return None
    output_prefix = asset_dir / output_stem
    cmd = ['pdftoppm', '-png', '-singlefile', '-r', '180', str(pdf_path), str(output_prefix)]
    subprocess.run(cmd, check=True)
    singlefile_path = asset_dir / f'{output_stem}.png'
    if singlefile_path.is_file():
        return singlefile_path
    matches = sorted(asset_dir.glob(f'{output_stem}*.png'))
    return matches[0] if matches else None


def image_tag(path, caption=None):
    if path is None:
        return '<p class="muted">ROC image not found.</p>'
    rel = os.path.relpath(path, project_root)
    caption_html = f'<figcaption>{esc(caption)}</figcaption>' if caption else ''
    return f'<figure><img src="{esc(rel)}" alt="{esc(caption or path.name)}">{caption_html}</figure>'


def roc_grid_for_final_selection(folder, prefix):
    pieces = ['<div class="roc-grid">']
    for dataset_name, _, roc_file in dataset_specs:
        png = pdf_to_png(Path(folder) / roc_file, f'{prefix}_{dataset_name.replace(" ", "_")}_roc')
        pieces.append(image_tag(png, f'{dataset_name} ROC'))
    pieces.append('</div>')
    return '\n'.join(pieces)


def roc_grid_for_soft_vote(prefix='fusion_soft_vote'):
    pieces = ['<div class="roc-grid">']
    for dataset_name, pdf_path in fusion_soft_vote['roc_paths'].items():
        png = pdf_to_png(pdf_path, f'{prefix}_{dataset_name.replace(" ", "_")}_roc')
        pieces.append(image_tag(png, f'{dataset_name} ROC'))
    pieces.append('</div>')
    return '\n'.join(pieces)

print('Helper functions ready.')


Helper functions ready.


In [3]:
# ============================================================
# 3. Build HTML sections
# ============================================================

sections = []
overview_rows = []

for branch in branches:
    folder = branch['folder']
    cv_manifest = read_json(folder / 'cv_final_selection_manifest.json')
    settings = selected_settings_from_manifest(cv_manifest)
    algorithm = majority_algorithm_from_manifest(cv_manifest, fallback='NA')
    metrics_df, source_df = collect_final_selection_metrics(folder)

    overview_rows.append({'Branch': branch['name'], 'Algorithm': algorithm, 'Output folder': str(folder)})

    section_html = f'''
    <section>
      <h2>{esc(branch['name'])}</h2>
      <p>{esc(branch['description'])}</p>
      <p><strong>CV-selected algorithm:</strong> {esc(algorithm)}</p>
      <h3>Selected CV setting(s)</h3>
      {describe_settings(settings)}
      <h3>Final-selection metrics</h3>
      {dataframe_to_html(metrics_df)}
      <h3>ROC curves</h3>
      {roc_grid_for_final_selection(folder, branch['name'].lower().replace(' ', '_').replace('-', '_'))}
      <details>
        <summary>Source files</summary>
        {dataframe_to_html(source_df)}
      </details>
    </section>
    '''
    sections.append(section_html)

soft_metrics_df = collect_soft_vote_metrics(fusion_soft_vote['metrics_path'])
overview_rows.append({'Branch': 'Fusion soft vote', 'Algorithm': 'N/A', 'Output folder': str(fusion_soft_vote['metrics_path'].parent)})
sections.append(f'''
<section>
  <h2>Fusion</h2>
  <h3>Fusion soft vote</h3>
  <p>Soft vote uses the arithmetic mean of the selected clinical, whole-image radiomics, habitat radiomics, and DL probability outputs.</p>
  <p><strong>Algorithm:</strong> N/A</p>
  <h4>Metrics</h4>
  {dataframe_to_html(soft_metrics_df)}
  <h4>ROC curves</h4>
  {roc_grid_for_soft_vote()}
</section>
''')

if len(stacking_folders) == 0:
    sections.append('<section><h2>Fusion stacking</h2><p class="muted">No stacking final-selection folder was found.</p></section>')
else:
    for stacking_folder in stacking_folders:
        cv_manifest = read_json(stacking_folder / 'cv_final_selection_manifest.json')
        settings = selected_settings_from_manifest(cv_manifest)
        algorithm = majority_algorithm_from_manifest(cv_manifest, fallback=stacking_folder.name)
        metrics_df, source_df = collect_final_selection_metrics(stacking_folder)
        overview_rows.append({'Branch': f'Fusion stacking ({stacking_folder.name})', 'Algorithm': algorithm, 'Output folder': str(stacking_folder)})
        sections.append(f'''
        <section>
          <h2>Fusion stacking</h2>
          <h3>{esc(stacking_folder.name)}</h3>
          <p>Stacking uses the selected fusion probability features as input to the meta-learner.</p>
          <p><strong>CV-selected algorithm:</strong> {esc(algorithm)}</p>
          <h4>Selected CV setting(s)</h4>
          {describe_settings(settings)}
          <h4>Final-selection metrics</h4>
          {dataframe_to_html(metrics_df)}
          <h4>ROC curves</h4>
          {roc_grid_for_final_selection(stacking_folder, 'fusion_stacking_' + stacking_folder.name.lower())}
          <details>
            <summary>Source files</summary>
            {dataframe_to_html(source_df)}
          </details>
        </section>
        ''')

overview_df = pd.DataFrame(overview_rows)
display(overview_df)
print('Sections prepared:', len(sections))


,Branch,Algorithm,Output folder
0,Clinical,LR,/host/d/projects/Habitats/models/Prognosis/cli...
1,Whole-image radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/who...
2,Habitat radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/hab...
3,DL 3D ML,LR,/host/d/projects/Habitats/models/Prognosis/dl_...
4,Fusion soft vote,N/A,/host/d/projects/Habitats/models/Prognosis/fusion
5,Fusion stacking (RF),RF,/host/d/projects/Habitats/models/Prognosis/fus...


Sections prepared: 6


In [4]:
# ============================================================
# 4. Render and save summary.html
# ============================================================

generated_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

style = """
:root { color-scheme: light; --text: #1f2933; --muted: #64748b; --border: #d9e2ec; --bg: #ffffff; --soft: #f8fafc; --accent: #1f77b4; }
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--text); font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif; line-height: 1.58; }
main { max-width: 1180px; margin: 0 auto; padding: 42px 48px 80px; }
h1 { font-size: 34px; line-height: 1.2; margin: 0 0 20px; border-bottom: 2px solid var(--border); padding-bottom: 16px; }
h2 { font-size: 27px; margin: 46px 0 16px; padding-top: 8px; border-top: 1px solid var(--border); }
h3 { font-size: 21px; margin: 30px 0 12px; }
h4 { font-size: 18px; margin: 24px 0 10px; }
p, ul, ol { margin: 10px 0 16px; }
code { background: #eef2f7; border-radius: 4px; padding: 2px 5px; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 0.92em; }
table { border-collapse: collapse; width: 100%; margin: 14px 0 24px; font-size: 14px; }
th, td { border: 1px solid var(--border); padding: 8px 10px; vertical-align: top; }
th { background: var(--soft); font-weight: 650; }
tr:nth-child(even) td { background: #fbfdff; }
a { color: var(--accent); text-decoration: none; }
a:hover { text-decoration: underline; }
figure { margin: 0; }
figcaption { color: var(--muted); font-size: 13px; margin-top: 4px; }
img { display: block; width: 100%; height: auto; border: 1px solid var(--border); border-radius: 8px; background: white; }
.roc-grid { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 12px 0 22px; }
.muted { color: var(--muted); }
details { border: 1px solid var(--border); border-radius: 8px; padding: 10px 14px; background: #fcfdff; margin: 14px 0 24px; }
summary { cursor: pointer; font-weight: 650; }
@media (max-width: 880px) { main { padding: 26px 18px 54px; } .roc-grid { grid-template-columns: 1fr; } table { font-size: 12px; } }
@media print { main { max-width: none; padding: 24px; } h2 { break-before: page; } a { color: var(--text); } }
"""

html_doc = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Habitat / Radiomics / DL Experiment Summary</title>
<style>{style}</style>
</head>
<body>
<main>
<h1>Habitat / Radiomics / DL Experiment Summary</h1>
<p>Generated: <code>{esc(generated_time)}</code></p>
<p>This report refreshes the current final-selection results for the Prognosis task, including the latest fusion stacking final-selection output.</p>
<h2>Overview</h2>
{dataframe_to_html(overview_df)}
{''.join(sections)}
</main>
</body>
</html>
'''

summary_results_path.write_text(html_doc, encoding='utf-8')
summary_root_path.write_text(html_doc, encoding='utf-8')

print('Saved summary HTML:')
print(' ', summary_results_path)
print(' ', summary_root_path)


Saved summary HTML:
  /host/d/projects/Habitats/results/summary.html
  /host/d/projects/Habitats/summary.html
